In [1]:
import pandas as pd 
import yfinance as yf 
df = yf.download("AAPL", start="2020-01-01", end="2024-01-01")
df.reset_index(inplace=True)
df.to_csv("aapl.csv", index=False)
df = pd.read_csv("aapl.csv")

ModuleNotFoundError: No module named 'yfinance'

#1 Basic exploration. How many rows and columns does dataset have?
what are the column names and their data types?
are there any missing values? if so, which columns and why? 

In [ ]:
df.shape

df.info()  # Display information about the DataFrame, big picture 

df.isnull().sum() #specific check for missing values in each column (count)

<class 'pandas.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    1006 non-null   str  
 1   Close   1007 non-null   str  
 2   High    1007 non-null   str  
 3   Low     1007 non-null   str  
 4   Open    1007 non-null   str  
 5   Volume  1007 non-null   str  
dtypes: str(6)
memory usage: 47.3 KB


Date      1
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [ ]:
#all datatypes are string. convert price columns to numeric
for col in ['Close', 'High', 'Low', 'Open']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

#Volume to integer
df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')

# date to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    1006 non-null   datetime64[us]
 1   Close   1006 non-null   float64       
 2   High    1006 non-null   float64       
 3   Low     1006 non-null   float64       
 4   Open    1006 non-null   float64       
 5   Volume  1006 non-null   float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 47.3 KB


Date      1
Close     1
High      1
Low       1
Open      1
Volume    1
dtype: int64

In [ ]:
#now that there is one missing value in the same row, drop the row. 

df.dropna(inplace=True) #removes row with at least one missing value. leaves us with 1006 clean rows.
df.reset_index(drop=True, inplace=True) #reset row numbers to 0-1005

df.isnull().sum()

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

#2 Indexing and Selecting: Select only the date and close columns.
show the first 10 rows for just those two columns. 
show all rows where the closing price was above $150

In [ ]:
df[['Date','Close']].head(10)

,Date,Close
0,2020-01-02,72.333893
1,2020-01-03,71.630623
2,2020-01-06,72.201408
3,2020-01-07,71.861839
4,2020-01-08,73.017822
5,2020-01-09,74.568802
6,2020-01-10,74.737381
7,2020-01-13,76.334084
8,2020-01-14,75.303329
9,2020-01-15,74.980606


In [ ]:
df.loc[df['Close'] >= 150]

,Date,Close,High,Low,Open,Volume
422,2021-09-03,150.629562,150.951714,149.448339,150.102399,57808700.0
423,2021-09-07,152.962708,153.519141,150.717416,151.283621,82278300.0
424,2021-09-08,151.420334,153.304417,150.317209,153.245847,74420200.0
425,2021-09-09,150.405045,152.396511,150.287889,151.791264,57305700.0
474,2021-11-17,150.057510,151.533737,147.613418,147.623189,88807000.0
...,...,...,...,...,...,...
1001,2023-12-22,191.433090,193.222829,190.810137,192.995392,37149600.0
1002,2023-12-26,190.889236,191.719831,190.671698,191.442966,28919300.0
1003,2023-12-27,190.988129,191.334217,188.951188,190.335527,48087700.0
1004,2023-12-28,191.413330,192.481244,191.007915,191.967060,34049900.0


#3 Summary statistics: what was the mean closing price over the entire period?
what was the highest closing price and on what date did it occur? 
what was the lowest closing price and on what date?

In [ ]:
df.Close.mean()


np.float64(137.83100558938847)

In [ ]:
#highest closing price and date 

max = df['Close'].max()
df.loc[df['Close'].idxmax(), ['Close', 'Date']]

Close             195.892624
Date     2023-12-14 00:00:00
Name: 995, dtype: object

In [ ]:
#lowest closing price and date

low = df['Close'].min()
df.loc[df['Close'].idxmin(), ['Close', 'Date']]

Close                54.1637
Date     2020-03-23 00:00:00
Name: 55, dtype: object

#4 Creating new columns: 
column daily_return- calculates the percentage change in closing price from one day to next (pct_change)
column price_range that is difference between High and Low for each day

In [ ]:
# % change in closing price from one day to the next.
df['daily_return'] = df['Close'].pct_change() * 100
df['daily_return'].round(2).head(10)

0     NaN
1   -0.97
2    0.80
3   -0.47
4    1.61
5    2.12
6    0.23
7    2.14
8   -1.35
9   -0.43
Name: daily_return, dtype: float64

In [ ]:
#difference between High and Low for each day.
df['price_range'] = df['High'] - df['Low']
df['price_range'].round(2).head(10)

0    1.30
1    0.98
2    1.74
3    0.82
4    1.75
5    1.02
6    1.06
7    1.43
8    1.30
9    1.43
Name: price_range, dtype: float64

#5 Grouping: add column 'year'extracted from data column.
group year and compute the mean closing price for each year. 
which year had the highest average closing price?

In [ ]:
df['Year'] = df['Date'].dt.year
df['Year'].head(10)

0    2020
1    2020
2    2020
3    2020
4    2020
5    2020
6    2020
7    2020
8    2020
9    2020
Name: Year, dtype: int32

In [ ]:
df.groupby('Year')['Close'].mean().round(2)


Year
2020     92.35
2021    137.47
2022    151.81
2023    170.19
Name: Close, dtype: float64

#6 sorting and filtering: 
Find 5 days with highest daily return
Find 5 days with the lowest daily return 
Filter to show only rows from 2022

In [ ]:
# sort 5 highest daily returns
df.sort_values(by='daily_return', ascending=False)[['Date', 'daily_return']].head(5)

,Date,daily_return
49,2020-03-13,11.980818
146,2020-07-31,10.468905
56,2020-03-24,10.032525
40,2020-03-02,9.310084
721,2022-11-10,8.897478


In [ ]:
# sort 5 lowest daily returns
df.sort_values(by='daily_return', ascending=True)[['Date', 'daily_return']].head(5)

,Date,daily_return
50,2020-03-16,-12.864673
48,2020-03-12,-9.875470
170,2020-09-03,-8.006079
45,2020-03-09,-7.909227
172,2020-09-08,-6.729496


In [ ]:
df.loc[df['Year'] == 2022]

,Date,Close,High,Low,Open,Volume,daily_return,price_range,Year
505,2022-01-03,177.939728,178.790282,173.735900,173.853212,104487900.0,2.500425,5.054382,2022
506,2022-01-04,175.681381,178.848931,175.114350,178.545866,99310400.0,-1.269164,3.734581,2022
507,2022-01-05,171.008286,176.140880,170.734548,175.593406,94537600.0,-2.659983,5.406332,2022
508,2022-01-06,168.153580,171.379785,167.801630,168.837923,96904000.0,-1.669338,3.578155,2022
509,2022-01-07,168.319794,170.245740,167.205288,169.023694,86709100.0,0.098847,3.040452,2022
...,...,...,...,...,...,...,...,...,...
751,2022-12-23,129.659378,130.210030,127.476427,128.735063,63814900.0,-0.279831,2.733603,2022
752,2022-12-27,127.859917,129.216891,126.571782,129.187392,69007800.0,-1.387837,2.645109,2022
753,2022-12-28,123.936539,128.843259,123.769378,127.505955,85438400.0,-3.068497,5.073882,2022
754,2022-12-29,127.446960,128.302436,125.598338,125.853994,75703700.0,2.832435,2.704098,2022
